# Flocking: Craig Reynolds' Boids

This notebook explores **Craig Reynolds' flocking model** (1986), commonly known as *Boids*. The core insight is that complex group behavior — flocking birds, schooling fish, herding animals — emerges from just three simple local rules applied by each individual agent:

| Rule | Description |
|------|-------------|
| **Separation** | Steer away from neighbors that are too close |
| **Alignment** | Match velocity with nearby neighbors |
| **Cohesion** | Move toward the center of nearby neighbors |

Each agent ("boid") only perceives its **local neighborhood** via its two proximeter sensors (left and right), and that is enough to produce realistic-looking flocks.

## 1. Starting the simulation

The `flocking` scene contains **12 boid agents** with a narrow forward field of view and short sensing range, similar to the `boyds` scene. Each agent uses:
- **LOVE** toward other boids — cohesion: agents are attracted to each other and slow down when close.
- Physical collision — separation: agents cannot overlap.
- Alignment emerges as agents follow whoever is directly ahead.

Run the cell below to connect to the simulation.

In [1]:
from vivarium.controllers import VivariumController

controller = VivariumController.start_session(scene_name="flocking")

INFO:vivarium.controllers.vivarium_controller:Server already running with scene 'flocking'. Connecting.
INFO:vivarium.controllers.vivarium_controller:Controller thread started on client
INFO:vivarium.controllers.vivarium_controller:Controller thread is already running
INFO:vivarium.controllers.vivarium_controller:VivariumController session 'flocking' is started


## 2. Exploring the agents

There are 12 boid agents in the scene. Let's inspect the first one.

In [2]:
print(f"Number of boid agents: {len(controller.agents)}")
controller.agents[0].print_infos()

Number of boid agents: 12
Entity Overview:
--------------------
Type: agents
Subtype: boid
Idx: 0
Exists: True
Position: x=33.81, y=65.17
Diameter: 2.00
Color: #4488ff

Sensors: Left=0.00, Right=0.15
Motors: Left=1.00, Right=0.85



Each agent has one behavior slot. Let's check what behavior is active on the first agent:

In [3]:
agent = controller.agents[0]
for i, beh in enumerate(agent.behaviors):
    print(f"Behavior slot {i}: {beh.label.name}, sensing: {beh.sensed}")

## 3. Observing the flock

The simulation is already running. Look at the web interface to observe the 12 boids. You should see:
- **Cohesion**: boids move toward each other and form groups
- **Separation**: collision forces prevent piling up
- **Alignment**: agents following the same neighbor end up moving in the same direction

Read the current proximeter values of a boid to confirm it is sensing its neighbors:

In [4]:
left, right = agent.proximeters()
print(f"Boid 0 proximeters  — Left: {left:.3f}  Right: {right:.3f}")

Boid 0 proximeters  — Left: 0.000  Right: 0.014


## 4. Tuning the flock parameters

### 4a. Sensing range

The sensing range (`proxs_dist_max`) controls how far each boid can perceive its neighbors. The default is 15. Increasing it causes agents to react to more distant neighbors.

In [5]:
# Increase sensing range
for boid in controller.agents:
    boid.proxs_dist_max = 30.0

In [6]:
# Reset to default
for boid in controller.agents:
    boid.proxs_dist_max = 15.0

### 4b. Coloring by role

Let's colour the first 3 boids red so we can follow them and see how they join or leave groups.

In [7]:
for boid in controller.agents[:3]:
    boid.color = 'red'

for boid in controller.agents[3:]:
    boid.color = '#4488ff'  # default blue

## 5. Custom Python flocking behavior

We can implement the three Reynolds rules more explicitly using a Python behavior function.
The function below combines a constant forward drive with separation (steer away from proximeter activation) and cohesion (steer toward the side with a neighbor):

In [8]:
def boids_behavior(agent,
                   base_speed=0.6,
                   separation_weight=0.8,
                   cohesion_weight=0.3):
    """Craig Reynolds Boids approximation using proximeters.

    Separation  — steer away from proximeter activation.
    Cohesion    — steer toward the side with stronger signal.
    Alignment   — emerges as agents follow their neighbors.

    Returns (left_motor, right_motor) both in [0, 1].
    """
    left_prox, right_prox = agent.proximeters()

    # Separation: back away from whatever is closest
    sep_left  = -separation_weight * left_prox
    sep_right = -separation_weight * right_prox

    # Cohesion: turn toward the side with a neighbor
    coh_left  = cohesion_weight * right_prox
    coh_right = cohesion_weight * left_prox

    left_motor  = base_speed + sep_left  + coh_left
    right_motor = base_speed + sep_right + coh_right

    return max(0.0, min(1.0, left_motor)), max(0.0, min(1.0, right_motor))

Attach this custom behavior to all 12 boids:

In [9]:
for boid in controller.agents:
    boid.attach_behavior(boids_behavior)

To go back to the built-in LOVE behavior:

In [ ]:
for boid in controller.agents:
    boid.detach_all_behaviors(stop_motors=True)

## 6. Experiments

**Q1:** What happens when you set `separation_weight = 0`? What about `cohesion_weight = 0`? Describe the observed behavior in each case.

In [ ]:
# your code here

*Double-click here and write your answer to Q1.*

**Q2:** Increase `base_speed` to `0.9`. How does speed affect the shape and stability of the flock? Why?

In [ ]:
# your code here

*Double-click here and write your answer to Q2.*

**Q3 (bonus):** Try widening the field of view by changing `proxs_cos_min` to `0.0`. Does the flock become more or less cohesive? Why?

In [ ]:
# Wider field of view
for boid in controller.agents:
    boid.proxs_cos_min = 0.0

# Restore default afterward
# for boid in controller.agents:
#     boid.proxs_cos_min = 0.9

*Double-click here and write your answer to Q3.*

## 7. Closing the session

When you are done, stop all behaviors and shut down the server.

In [ ]:
for boid in controller.agents:
    boid.detach_all_behaviors(stop_motors=True)

controller.simulator.simulation_running = False